# Chromadb를 활용한 DB SCHEMA 작성

#### 환경 설정

In [1]:
# 라이브러리와 한국어 임베딩 모델
import numpy as np
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
from typing import List
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

c:\Users\Playdata\Desktop\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3790.51it/s]


임베딩 모델 준비 완료 — 벡터 차원: 768


#### 데이터 가져오기

#### Langchain 활용 chromadb 구축

#### 청킹 데이터 3개 통합

In [2]:
from pathlib import Path


# 프로젝트 구조에 맞는 데이터 폴더
DATA_DIR = "../data/RAG/"

def load_documents(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = [
        Document(page_content=doc["page_content"], metadata=doc["metadata"])
        for doc in data
    ]

    return documents

guide_documents = load_documents(DATA_DIR + "maple_guides_documents_chunked.json")
jobs_documents = load_documents(DATA_DIR + "maple_jobs_documents.json")
items_documents = load_documents(DATA_DIR + "maple_items_documents.json")

for doc in guide_documents:
    doc.metadata['source'] = "guide"
for doc in jobs_documents:
    doc.metadata['source'] = "jobs"
for doc in items_documents:
    doc.metadata['source'] = "items"

all_documents = guide_documents + jobs_documents + items_documents

In [3]:

def build_maplestory_chromadb(
    chunked_documents: List[Document], 
    persist_directory: str = "../chroma_db",
    collection_name: str = "maplestory_guides"
) -> Chroma:
    """
    메이플스토리 청크 문서를 ChromaDB에 스키마를 맞춰 저장합니다.
    """
    # 1. 한국어 성능이 뛰어난 임베딩 모델 로드
    embeddings = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={'device': 'cpu'},  # GPU 환경일 경우 'cuda'
        encode_kwargs={'normalize_embeddings': True}  # 코사인 유사도 검색 최적화
    )
    
    # 2. 문서별 고유 ID 목록 생성 ({article_id}_chunk_{chunk_index})
    ids = []
    for idx, doc in enumerate(chunked_documents):
        # chunk_index가 없으면 현재 순번(idx) 부여
        chunk_index = doc.metadata.get('chunk_index', idx)
        ids.append(f"chunk_{chunk_index}")
        
    # 3. ChromaDB 생성 및 저장 (Cosine Distance 기준)
    vectorstore = Chroma.from_documents(
        documents=chunked_documents,
        embedding=embeddings,
        ids=ids,
        collection_name=collection_name,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}  # 유사도 계산 방식 설정 (cosine, l2, ip)
    )
    
    print(f"ChromaDB에 {len(chunked_documents)}개의 청크가 성공적으로 저장되었습니다.")
    return vectorstore

build_maplestory_chromadb(all_documents)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1875.46it/s]


ChromaDB에 3694개의 청크가 성공적으로 저장되었습니다.


#### Chromadb 연결

#### Chromadb Schema
```JSON
{
    "source": "guide",                  // 출처 (guide, patch_note, faq 등)
    "name": "보스 레이드: 자쿰 가이드",     // 문서 원본 제목
    "section_title": "보스/레이드",       // 카테고리 (필터링 핵심 키)
    "article_id": 101,                  // 원본 게시글 ID (Integer 또는 String)
    "board_id": 1,                      // 게시판 ID
    "url": "https://maplestory...",     // 출처 링크 (답변 시 참조 URL 제공용)
    "chunk_index": 0,                   // 문서 내 청크 순서
    "total_chunks": 3                   // 문서 전체 청크 수
}
```

#### 의미 기반 검색 (임베딩 코사인 유사도)

In [ ]:
# 질문도 문서와 같은 방식으로 임베딩한다
query = ""



query_emb = emb_model.encode([query], normalize_embeddings=True)

# 결과가 1×N 행렬이라 [0] 으로 한 줄을 꺼낸다
sims = cosine_similarity(query_emb, doc_emb)[0]

# 부호를 뒤집어 argsort 하면 내림차순
top3 = np.argsort(-sims)[:3]

print("[의미 기반 검색] Top-3:")
for i in top3:
    print(f"  유사도 {sims[i]:.3f}  |  {spots.loc[i, 'name']} ({spots.loc[i, 'type']})")

#### Top-K 검색

In [ ]:
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

res = travel_col.query(query_embeddings=query_emb, n_results=3)

print("질문:", query)

# 질문을 여러 개 넣을 수 있는 구조라 [0] 으로 한 겹 벗긴다
for doc_id, dist, meta in zip(
    res['ids'][0], res['distances'][0], res['metadatas'][0]
):
    print(f"  거리 {dist:.3f}  |  {doc_id}  {meta['name']} ({meta['type']})")

#### Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1) 클라이언트와 컬렉션
qclient = QdrantClient(':memory:')
dim = doc_emb.shape[1]   # Qdrant 는 차원을 미리 알려 줘야 한다

# create_collection 은 이미 있으면 에러 — 지우고 다시 만든다
if qclient.collection_exists('travel_guide'):
    qclient.delete_collection('travel_guide')

qclient.create_collection(
    'travel_guide',
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

# 2) 적재 — PointStruct 하나가 문서 하나 (id·vector·payload)
qclient.upsert('travel_guide', points=[
    PointStruct(
        id=i,
        vector=doc_emb[i],
        payload={'name': spots.loc[i, 'name'], 'type': spots.loc[i, 'type']},
    )
    for i in range(len(spots))
])

print("Qdrant 에 저장된 점 개수:", qclient.count("travel_guide").count)

# 3) 검색 — ChromaDB 와 같은 질문
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 질문 벡터를 리스트로 감싸지 않고 하나만 넘긴다
hits = qclient.query_points('travel_guide', query=query_emb[0], limit=3).points

print("\n[Qdrant] 질문:", query)

# score 는 Chroma 의 distance 와 반대 — 클수록 가깝다
for h in hits:
    print(f"  점수 {h.score:.3f}  |  {h.payload['name']} ({h.payload['type']})")